In [ ]:
# !pip install -q pypdf
# !pip install -q python-dotenv
# !pip install langchain
# !pip install llama-index-embeddings-langchain
# !pip install transformers
# !pip install -q llama-index
# %pip install llama-index-llms-llama-cpp

In [ ]:
# !pip install -q llama-index
# %pip install llama-index-llms-llama-cpp
# !pip install -U langchain-community
# %pip install llama-index-embeddings-huggingface


In [ ]:
# !pip install rouge-score nltk
# import nltk
# nltk.download('punkt')


In [ ]:
# !pip install rouge-score nltk

# import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')


In [ ]:
# ================================
# ─ 1. Import Libraries
# ================================
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.llms.llama_cpp.llama_utils import messages_to_prompt, completion_to_prompt
from sklearn.metrics.pairwise import cosine_similarity

# NEW embedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# NEW: ROUGE + BLEU
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize


# ================================
# ─ 2. Sample Text About Cancer
# ================================
cancer_text = """
Cancer is a disease in which abnormal cells divide uncontrollably and can invade nearby tissues.
These abnormal cells may also spread to other parts of the body through the blood and lymph systems.
Cancer can start almost anywhere in the human body. There are many types of cancer including breast cancer,
lung cancer, prostate cancer, and blood cancers like leukemia.

Common symptoms of cancer include unexplained weight loss, fatigue, lumps, prolonged cough, and changes in bowel habits.
Treatment options include chemotherapy, radiation, surgery, immunotherapy, and targeted therapy.
Early detection significantly improves survival rates.
"""


# ================================
# ─ 3. Ground Truth Q&A
# ================================
test_questions = [
    "What is cancer?",
    "What are common symptoms of cancer?",
    "How can cancer spread in the body?",
]

ground_truth = {
    "What is cancer?":
        "Cancer is a disease where abnormal cells divide uncontrollably and invade nearby tissues.",

    "What are common symptoms of cancer?":
        "Common symptoms include weight loss, fatigue, lumps, cough, and changes in bowel habits.",

    "How can cancer spread in the body?":
        "Cancer can spread through the blood and lymphatic systems to other parts of the body.",
}


# ================================
# ─ 4. Embedding Similarity
# ================================
embed_model = HuggingFaceEmbedding(model_name="thenlper/gte-large")

def similarity(a, b):
    v1 = embed_model.get_text_embedding(a)
    v2 = embed_model.get_text_embedding(b)
    return cosine_similarity([v1], [v2])[0][0]


# ================================
# ─ 5. ROUGE Score
# ================================
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def rouge_score(pred, ref):
    scores = rouge.score(ref, pred)
    return scores["rouge1"].fmeasure, scores["rougeL"].fmeasure


# ================================
# ─ 6. BLEU Score
# ================================
def bleu_score(pred, ref):
    pred_tokens = word_tokenize(pred.lower())
    ref_tokens = [word_tokenize(ref.lower())]
    smooth = SmoothingFunction().method1
    return sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smooth)


# ================================
# ─ 7. Ask the LLM
# ================================
def ask_llm(index, question):
    response = index.as_query_engine().query(question)
    return str(response)


# ================================
# ─ 8. MAIN PIPELINE
# ================================
docs = [Document(text=cancer_text)]

Settings.chunk_size = 256
Settings.embed_model = embed_model
index = VectorStoreIndex.from_documents(docs)

# Load local model
llm = LlamaCPP(
    model_url='https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf',
    temperature=0.1,
    max_new_tokens=200,
    context_window=4096,
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    verbose=False
)
Settings.llm = llm

# Evaluate questions
final_scores = {}

for q in test_questions:
    answer = ask_llm(index, q)

    # COSINE SIMILARITY
    cos_sim = similarity(answer, ground_truth[q])

    # ROUGE
    rouge1, rougeL = rouge_score(answer, ground_truth[q])

    # BLEU
    bleu = bleu_score(answer, ground_truth[q])

    print("\n==============================")
    print("QUESTION:", q)
    print("GROUND TRUTH:", ground_truth[q])
    print("MODEL ANSWER:", answer)
    print(f"Cosine Similarity: {cos_sim}")
    print(f"ROUGE-1: {rouge1}")
    print(f"ROUGE-L: {rougeL}")
    print(f"BLEU: {bleu}")

    final_scores[q] = {
        "cosine": cos_sim,
        "rouge1": rouge1,
        "rougeL": rougeL,
        "bleu": bleu
    }

print("\n\n─ Final Scores:", final_scores)


##Note

Observation

“Cosine similarity captures meaning, so it remained high.
ROUGE and BLEU require exact word overlap and penalize paraphrasing, so they are lower.
This indicates the model answered correctly in meaning but not in wording.”